In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

#df = pd.read_csv(PROCESSED_DIR / "dawn_revenue_combined_clean.csv")
#df.head()

# Load the CSV file into a pandas DataFrame
candidate_paths = [
    RAW_DIR / "dawn_revenue_accounts_combined_history.csv",
    PROJECT_ROOT.parent / "data" / "dawn_revenue_accounts_combined_history.csv",
    PROJECT_ROOT.parent / "dawn_aggregate" / "data" / "dawn_revenue_accounts_combined_history.csv",
    PROCESSED_DIR / "dawn_revenue_accounts_combined_history.csv",
]

df_all = None
for path in candidate_paths:
    if path.exists():
        df_all = pd.read_csv(path)
        break

if df_all is None:
    raise FileNotFoundError(
        "Could not find 'dawn_revenue_accounts_combined_history.csv'. "
        f"Checked: {[str(p) for p in candidate_paths]}"
    )
df_all.head()

In [ ]:
print(df_all['fy'].unique())

In [ ]:
# Removing fy 2027 as it is not a complete year and may skew the analysis

df_all = df_all[df_all['fy'] != 2027]
print(f"Shape of DataFrame after dropping FY2027 data: {df_all.shape}")
display(df_all.head())

In [ ]:
# Dropping GL Number 42 as it is not relevant to the analysis

df_all_filtered = df_all[df_all['GL Number'] != 42]
print(f"Shape of DataFrame after dropping GL Number 42: {df_all_filtered.shape}")
display(df_all_filtered.head())

In [ ]:
# Convert Amount to numeric first
df_all['Amount'] = pd.to_numeric(df_all['Amount'], errors='coerce')

# Keep everything except GL 42 when Amount is greater than 0
df_all_filtered = df_all[
    ~(
        (df_all['GL Number'] == 42) &
        (df_all['Amount'] > 0)
    )
].copy()

print(f"Shape after dropping only GL 42 with Amount > 0: {df_all_filtered.shape}")
display(df_all_filtered.head())

## General Fund Calculation

In [ ]:
# filtering the DataFrame for BA Number 9999 and specific GL Numbers that we require for the analysis. These GL Numbers are related to the General Fund and are relevant to our analysis.

gl_numbers_to_keep = [3107, 3114, 3115, 3120, 3202, 3203, 3204, 3220, 3240, 3255, 3271, 3274, 3276, 3291, 4102, 4403, 4408, 4409]
df_filtered_ba_gl = df_all_filtered[(df_all_filtered['BA Number'] == 9999) & (df_all_filtered['GL Number'].isin(gl_numbers_to_keep))]
display(df_filtered_ba_gl.head())
print(f"Shape of DataFrame after filtering for BA Number 9999 and specific GL Numbers: {df_filtered_ba_gl.shape}")

In [ ]:
# filtering the DataFrame for BA Number 9999 and specific GL Numbers that we require for the analysis. These GL Numbers are related to the General Fund and are relevant to our analysis.

gl_numbers_to_keep = [3107, 3114, 3115, 3120, 3202, 3203, 3204, 3220, 3240, 3255, 3271, 3274, 3276, 3291, 4102, 4403, 4408, 4409]

mask = (
    (df_all_filtered['BA Number'] != 9999) |
    (
        (df_all_filtered['BA Number'] == 9999) &
        (df_all_filtered['GL Number'].isin(gl_numbers_to_keep))
    )
)

df_filtered = df_all_filtered.loc[mask].copy()

display(df_filtered.head())
print(f"Shape after keeping all BA values and limiting only BA 9999 GL codes: {df_filtered.shape}")

In [ ]:
# Now I want to keep only the rows where the 'BA Nnumber' column is 9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999.    
# I will filter the DataFrame accordingly.
ba_numbers_to_keep = [9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999]

df_filtered_ba = df_filtered[df_filtered['BA Number'].isin(ba_numbers_to_keep)]
display(df_filtered_ba.head())
print(f"Shape of DataFrame after filtering for specific BA Numbers: {df_filtered_ba.shape}")


In [ ]:
import plotly.express as px

# Group by fiscal year and sum the amount
#df_gf = df_filtered_ba.groupby('fy')['Amount'].sum().reset_index()



df_gf = (
    df_filtered_ba.assign(Amount=pd.to_numeric(df_filtered_ba['Amount'], errors='coerce'))
    .groupby('fy')['Amount']
    .sum()
    .reset_index()
)


# Create an interactive line plot
fig = px.line(
    df_gf,
    x='fy',
    y='Amount',
    title='Total Amount by Fiscal Year',
    labels={'fy': 'Fiscal Year', 'Amount': 'Total Amount'}
)

fig.update_traces(mode='lines+markers')
fig.update_layout(
    hovermode='x unified',
    xaxis_title='Fiscal Year',
    yaxis_title='Total Amount',
    xaxis=dict(tickmode='linear', dtick=1) # Ensure yearly ticks
)

fig.show()

In [ ]:
df_gf_table = df_filtered_ba.groupby('fy')['Amount'].sum().reset_index()
df_gf_table.columns = ['Fiscal Year', 'Total Amount']

pd.options.display.float_format = '{:,.2f}'.format
display(df_gf_table)
pd.options.display.float_format = None # Reset to default for other outputs

In [ ]:
df_2026 = df_filtered_ba[df_filtered_ba['fy'] == 2026]
df_2026_by_ba_gl = df_2026.groupby(['BA Number', 'GL Number'])['Amount'].sum().reset_index()
df_2026_by_ba_gl.columns = ['BA Number', 'GL Number', 'Total Amount']

# Format the 'Total Amount' column for better readability
pd.options.display.float_format = '{:,.2f}'.format
display(df_2026_by_ba_gl)
pd.options.display.float_format = None

In [ ]:
df_2026_table = (
    df_2026.groupby(['BA Number', 'GL Number'])['Amount']
    .sum()
    .sort_index()
)

pd.options.display.float_format = '{:,.2f}'.format
display(df_2026_table)
pd.options.display.float_format = None

In [ ]:
import pandas as pd
import plotly.express as px

ba_numbers_to_keep = [9010, 9040, 9050, 9090, 9130, 9360, 9600, 9705, 9707, 9741, 9748, 9752, 9999]

df_2026 = df_filtered[
    (df_filtered['fy'] == 2026) &
    (df_filtered['BA Number'].isin(ba_numbers_to_keep))
].copy()

df_2026['Amount'] = pd.to_numeric(df_2026['Amount'], errors='coerce')

plot_df = (
    df_2026.groupby(['BA Number', 'GL Number'], as_index=False)['Amount']
    .sum()
    .sort_values(['BA Number', 'GL Number'])
)

# One interactive chart per BA with values shown on each bar
for ba in sorted(plot_df['BA Number'].unique()):
    ba_df = plot_df[plot_df['BA Number'] == ba].copy()

    fig = px.bar(
        ba_df,
        x='GL Number',
        y='Amount',
        title=f'FY 2026 Total Amount by GL for BA {ba}',
        labels={'GL Number': 'GL Number', 'Amount': 'Total Amount'},
        color='Amount',
        color_continuous_scale='Viridis',
        text='Amount'
    )

    fig.update_traces(
        texttemplate='%{text:,.2f}',
        textposition='outside',
        cliponaxis=False
    )

    fig.update_layout(
        xaxis=dict(type='category'),
        template='plotly_white',
        hovermode='closest',
        yaxis_title='Total Amount',
        xaxis_title='GL Number'
    )
    fig.show()